# Medical Appointment No-Show Analysis

This notebook analyzes the Kaggle medical appointment no-show dataset and focuses on which
categories predict higher missed-appointment rates. The target is `No-show = Yes`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_FILE = Path("noshowappointments-kagglev2-may-2016.csv")
pd.set_option("display.max_columns", 50)

In [2]:
raw = pd.read_csv(DATA_FILE)
raw.shape, raw.head()

((110527, 14),
       PatientId  AppointmentID Gender          ScheduledDay  \
 0  2.987250e+13        5642903      F  2016-04-29T18:38:08Z   
 1  5.589978e+14        5642503      M  2016-04-29T16:08:27Z   
 2  4.262962e+12        5642549      F  2016-04-29T16:19:04Z   
 3  8.679512e+11        5642828      F  2016-04-29T17:29:31Z   
 4  8.841186e+12        5642494      F  2016-04-29T16:07:23Z   
 
          AppointmentDay  Age      Neighbourhood  Scholarship  Hipertension  \
 0  2016-04-29T00:00:00Z   62    JARDIM DA PENHA            0             1   
 1  2016-04-29T00:00:00Z   56    JARDIM DA PENHA            0             0   
 2  2016-04-29T00:00:00Z   62      MATA DA PRAIA            0             0   
 3  2016-04-29T00:00:00Z    8  PONTAL DE CAMBURI            0             0   
 4  2016-04-29T00:00:00Z   56    JARDIM DA PENHA            0             1   
 
    Diabetes  Alcoholism  Handcap  SMS_received No-show  
 0         0           0        0             0      No  
 1     

In [3]:
df = raw.rename(columns={
    "PatientId": "patient_id",
    "AppointmentID": "appointment_id",
    "Gender": "gender",
    "ScheduledDay": "scheduled_day",
    "AppointmentDay": "appointment_day",
    "Age": "age",
    "Neighbourhood": "neighbourhood",
    "Scholarship": "scholarship",
    "Hipertension": "hypertension",
    "Diabetes": "diabetes",
    "Alcoholism": "alcoholism",
    "Handcap": "handicap",
    "SMS_received": "sms_received",
    "No-show": "no_show_raw",
})
df["scheduled_day"] = pd.to_datetime(df["scheduled_day"], utc=True)
df["appointment_day"] = pd.to_datetime(df["appointment_day"], utc=True)
df["appointment_date"] = df["appointment_day"].dt.normalize()
df["scheduled_date"] = df["scheduled_day"].dt.normalize()
df["wait_days"] = (df["appointment_date"] - df["scheduled_date"]).dt.days
df["no_show"] = df["no_show_raw"].eq("Yes").astype(int)

quality = {
    "rows_before": len(df),
    "negative_age_rows": int((df["age"] < 0).sum()),
    "negative_wait_rows": int((df["wait_days"] < 0).sum()),
}
df = df.loc[df["age"].ge(0) & df["wait_days"].ge(0)].copy()
quality["rows_after"] = len(df)
quality

{'rows_before': 110527,
 'negative_age_rows': 1,
 'negative_wait_rows': 5,
 'rows_after': 110521}

In [4]:
wait_order = ["Same day", "1 day", "2-3 days", "4-7 days", "8-14 days", "15-30 days", "31+ days"]
age_order = ["0-12 child", "13-17 teen", "18-29 young adult", "30-44 adult", "45-59 midlife", "60-74 senior", "75+ older adult"]

df["wait_bucket"] = pd.cut(df["wait_days"], [-1, 0, 1, 3, 7, 14, 30, np.inf], labels=wait_order).astype(str)
df["age_group"] = pd.cut(df["age"], [-1, 12, 17, 29, 44, 59, 74, np.inf], labels=age_order).astype(str)
df["appointment_weekday"] = df["appointment_day"].dt.day_name()
df["scheduled_weekday"] = df["scheduled_day"].dt.day_name()
df["scheduled_hour"] = df["scheduled_day"].dt.hour
df["condition_count"] = (
    df["hypertension"].clip(0, 1)
    + df["diabetes"].clip(0, 1)
    + df["alcoholism"].clip(0, 1)
    + df["handicap"].gt(0).astype(int)
)
df["condition_count_bucket"] = pd.cut(
    df["condition_count"], [-1, 0, 1, 2, 4],
    labels=["0 conditions", "1 condition", "2 conditions", "3+ conditions"]
).astype(str)
df["scholarship_label"] = np.where(df["scholarship"].eq(1), "Scholarship: yes", "Scholarship: no")
df["sms_received_label"] = np.where(df["sms_received"].eq(1), "SMS received", "No SMS")

overall_rate = df["no_show"].mean()
overall_rate

np.float64(0.2018982817745044)

In [5]:
def summarize(column):
    return (
        df.groupby(column, observed=True)
        .agg(appointments=("appointment_id", "size"), no_shows=("no_show", "sum"), no_show_rate=("no_show", "mean"), median_wait_days=("wait_days", "median"))
        .assign(lift_vs_average=lambda x: x["no_show_rate"] / overall_rate - 1)
        .sort_values("no_show_rate", ascending=False)
    )

summarize("wait_bucket").loc[wait_order]

,appointments,no_shows,no_show_rate,median_wait_days,lift_vs_average
wait_bucket,,,,,
Same day,38562,1792,0.046471,0.0,-0.769832
1 day,5213,1113,0.213505,1.0,0.057486
2-3 days,9462,2246,0.237371,2.0,0.175694
4-7 days,17510,4413,0.252027,6.0,0.248289
8-14 days,12025,3664,0.304699,11.0,0.509169
15-30 days,17371,5661,0.325888,21.0,0.614120
31+ days,10378,3425,0.330025,40.0,0.634611


In [6]:
px.bar(
    summarize("wait_bucket").loc[wait_order].reset_index(),
    x="wait_bucket",
    y="no_show_rate",
    text="appointments",
    title="No-show rate rises sharply as lead time increases",
    labels={"no_show_rate": "No-show rate", "wait_bucket": "Lead time"},
).update_yaxes(tickformat=".0%")

In [7]:
summarize("age_group").loc[age_order]

,appointments,no_shows,no_show_rate,median_wait_days,lift_vs_average
age_group,,,,,
0-12 child,21035,4306,0.204706,3.0,0.013909
13-17 teen,6343,1690,0.266435,3.0,0.319652
18-29 young adult,16729,4122,0.246398,4.0,0.220409
30-44 adult,22021,4805,0.218201,4.0,0.080746
45-59 midlife,23221,4150,0.178718,4.0,-0.114814
60-74 senior,15237,2291,0.150358,4.0,-0.255280
75+ older adult,5935,950,0.160067,4.0,-0.207188


In [8]:
key_categories = {
    "SMS reminder": "sms_received_label",
    "Scholarship / welfare": "scholarship_label",
    "Condition count": "condition_count_bucket",
    "Appointment weekday": "appointment_weekday",
    "Neighborhood": "neighbourhood",
}

category_tables = {}
for label, column in key_categories.items():
    table = summarize(column)
    if column == "neighbourhood":
        table = table.query("appointments >= 500").head(12)
    category_tables[label] = table
category_tables["SMS reminder"], category_tables["Scholarship / welfare"], category_tables["Neighborhood"]

(                    appointments  no_shows  no_show_rate  median_wait_days  \
 sms_received_label                                                           
 SMS received               35482      9784      0.275745              14.0   
 No SMS                     75039     12530      0.166980               0.0   
 
                     lift_vs_average  
 sms_received_label                   
 SMS received               0.365764  
 No SMS                    -0.172951  ,
                    appointments  no_shows  no_show_rate  median_wait_days  \
 scholarship_label                                                           
 Scholarship: yes          10861      2578      0.237363               4.0   
 Scholarship: no           99660     19736      0.198033               4.0   
 
                    lift_vs_average  
 scholarship_label                   
 Scholarship: yes          0.175657  
 Scholarship: no          -0.019143  ,
                    appointments  no_shows  no_show_rate  

In [9]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

categorical_features = [
    "gender", "age_group", "wait_bucket", "appointment_weekday", "scheduled_weekday",
    "neighbourhood", "scholarship_label", "sms_received_label"
]
numeric_features = ["scheduled_hour", "condition_count"]
X = df[categorical_features + numeric_features]
y = df["no_show"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

model = Pipeline([
    ("preprocessor", ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", make_ohe(), categorical_features),
    ])),
    ("model", LogisticRegression(max_iter=1200, solver="lbfgs")),
])
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)[:, 1]

metrics = {
    "roc_auc": roc_auc_score(y_test, probabilities),
    "average_precision": average_precision_score(y_test, probabilities),
    "brier_score": brier_score_loss(y_test, probabilities),
    "test_base_rate": y_test.mean(),
}
metrics

{'roc_auc': 0.7270620629230518,
 'average_precision': 0.35494212177087286,
 'brier_score': 0.14470233272360888,
 'test_base_rate': np.float64(0.20191089718070285)}

In [10]:
df["predicted_no_show_probability"] = model.predict_proba(X)[:, 1]
risk_deciles = pd.DataFrame({"actual": df["no_show"], "predicted": df["predicted_no_show_probability"]})
risk_deciles["decile"] = pd.qcut(risk_deciles["predicted"].rank(method="first"), 10, labels=[f"D{i}" for i in range(1, 11)])
decile_summary = risk_deciles.groupby("decile", observed=True).agg(
    appointments=("actual", "size"),
    observed_no_show=("actual", "mean"),
    predicted_no_show=("predicted", "mean"),
)
decile_summary

,appointments,observed_no_show,predicted_no_show
decile,,,
D1,11053,0.027594,0.030233
D2,11052,0.037821,0.042669
D3,11052,0.054017,0.053860
D4,11052,0.125769,0.113170
D5,11052,0.201502,0.198056
D6,11052,0.230908,0.236940
D7,11052,0.268277,0.271289
D8,11052,0.309808,0.306746
D9,11052,0.344915,0.347563


In [11]:
px.line(
    decile_summary.reset_index(),
    x="decile",
    y=["observed_no_show", "predicted_no_show"],
    markers=True,
    title="Risk deciles separate lower-risk and higher-risk appointments",
    labels={"value": "No-show rate", "decile": "Predicted risk decile", "variable": "Series"},
).update_yaxes(tickformat=".0%")

## Recommended operational response

- **Long lead times**: Same-day appointments missed at 4.6%, while 31+ day waits missed at 33.0%. Protect near-term access, fill cancellations from a waitlist, and require active confirmation for appointments booked more than a week out.
- **Reminder design**: SMS-received appointments show 27.6% no-show versus 16.7% without SMS; this is likely confounded by longer waits. Do not interpret this as SMS causing no-shows. Use timed, two-way reminders: booking confirmation, 72-hour confirmation, day-before reminder, and easy reschedule links.
- **Young and socially vulnerable groups**: The highest age band is 13-17 teen at 26.6%; scholarship patients missed at 23.7% versus 19.8% for non-scholarship patients. Add friction-reducing support: flexible slots, transport guidance, caregiver contact options, and low-penalty rescheduling instead of punitive cancellation policies.
- **Neighborhood operations**: Among neighborhoods with at least 500 appointments, SANTOS DUMONT has the highest observed no-show rate at 28.9%. Target outreach geographically: adjust reminder channels, partner with local primary-care teams, and consider transport or telehealth alternatives in persistent hot spots.
- **Risk scoring**: The interpretable logistic model reaches ROC AUC 0.728; its highest-risk decile misses at 41.7%. Use risk tiers for operational triage, not denial of care: extra confirmations, standby lists, and compassionate rescheduling for high-risk appointments.

The model should be used to add support and improve scheduling operations, not to deny care.